### Representación ampliada

| Variable                     | Qué representa                                                       |
| ---------------------------- | -------------------------------------------------------------------- |
| `recency`                    | días desde la última compra                                          |
| `frequency`                  | número de ocasiones de compra del cliente                            |
| `monetary`                   | gasto total realizado                                                |
| `avg_basket_size`            | tamaño medio de cesta por ocasión de compra                          |
| `std_basket_size`            | variabilidad del tamaño de cesta                                     |
| `avg_price`                  | precio medio de los artículos comprados                              |
| `std_price`                  | variabilidad del precio de los artículos comprados                   |
| `avg_days_between_purchases` | tiempo medio entre ocasiones de compra                               |
| `std_days_between_purchases` | regularidad temporal de las compras                                  |
| `unique_articles`            | número de artículos diferentes comprados                             |
| `unique_product_types`       | variedad de tipos de producto comprados                              |
| `unique_product_groups`      | variedad de grupos de producto comprados                             |
| `unique_sections`            | variedad de secciones de producto                                    |
| `unique_garment_groups`      | variedad de grupos de prendas                                        |
| `article_variety_ratio`      | proporción de artículos distintos respecto al total comprado         |
| `channel_2_ratio`            | proporción de compras realizadas mediante el canal codificado como 2 |


In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../data/hym")
DATA_DIR_CLEAN = Path("../../data_without_nulls/hym")

In [4]:
hm_extended = pd.read_csv(
    "../../simple_datasets/hym_model_simple.csv"
)

# Solo cargamos las columnas que necesitamos para ahorrar memoria
transactions = pd.read_csv(
    DATA_DIR / "transactions_train.csv",
    usecols=[
        "t_dat",
        "customer_id",
        "article_id",
        "price",
        "sales_channel_id"
    ]
)

articles = pd.read_csv(
    DATA_DIR_CLEAN / "articles.csv",
    usecols=[
        "article_id",
        "product_type_no",
        "product_group_name",
        "section_no",
        "garment_group_no"
    ]
)

transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

### Tamaño de cesta

In [5]:
basket_per_purchase = (
    transactions
    .groupby(["customer_id", "t_dat"])
    .size()
    .reset_index(name="basket_size")
)

basket_features = (
    basket_per_purchase
    .groupby("customer_id")["basket_size"]
    .agg(
        avg_basket_size="mean",
        std_basket_size="std"
    )
    .reset_index()
)

# Si un cliente solo tiene una ocasión de compra,
# la desviación estándar no puede calcularse.
basket_features["std_basket_size"] = (
    basket_features["std_basket_size"]
    .fillna(0)
)

### Precio medio y variabilidad del precio

In [6]:
price_features = (
    transactions
    .groupby("customer_id")["price"]
    .agg(
        avg_price="mean",
        std_price="std"
    )
    .reset_index()
)

price_features["std_price"] = (
    price_features["std_price"]
    .fillna(0)
)

### Regularidad temporal

In [7]:
purchase_dates = (
    transactions[
        ["customer_id", "t_dat"]
    ]
    .drop_duplicates()
    .sort_values(["customer_id", "t_dat"])
)

In [8]:
purchase_dates["days_between_purchases"] = (
    purchase_dates
    .groupby("customer_id")["t_dat"]
    .diff()
    .dt.days
)

In [9]:
temporal_features = (
    purchase_dates
    .groupby("customer_id")["days_between_purchases"]
    .agg(
        avg_days_between_purchases="mean",
        std_days_between_purchases="std"
    )
    .reset_index()
)

In [10]:
temporal_features[
    [
        "avg_days_between_purchases",
        "std_days_between_purchases"
    ]
] = temporal_features[
    [
        "avg_days_between_purchases",
        "std_days_between_purchases"
    ]
].fillna(0)

### Información de los artículos

In [11]:
purchases = transactions[
    ["customer_id", "article_id"]
].merge(
    articles,
    on="article_id",
    how="left"
)

### Artículos diferentes

In [12]:
unique_articles = (
    purchases
    .groupby("customer_id")["article_id"]
    .nunique()
    .reset_index(name="unique_articles")
)

### Tipos de productos diferentes

In [13]:
unique_product_types = (
    purchases
    .groupby("customer_id")["product_type_no"]
    .nunique()
    .reset_index(name="unique_product_types")
)

### Grupos de producto diferentes

In [14]:
unique_product_groups = (
    purchases
    .groupby("customer_id")["product_group_name"]
    .nunique()
    .reset_index(name="unique_product_groups")
)

### Secciones diferentes

In [15]:
unique_sections = (
    purchases
    .groupby("customer_id")["section_no"]
    .nunique()
    .reset_index(name="unique_sections")
)

### Grupos de prendas diferentes

In [16]:
unique_garment_groups = (
    purchases
    .groupby("customer_id")["garment_group_no"]
    .nunique()
    .reset_index(name="unique_garment_groups")
)

### Ratio de variedad de artículos

In [17]:
total_articles = (
    transactions
    .groupby("customer_id")
    .size()
    .reset_index(name="total_articles")
)

article_variety = total_articles.merge(
    unique_articles,
    on="customer_id",
    how="left"
)

article_variety["article_variety_ratio"] = (
    article_variety["unique_articles"]
    / article_variety["total_articles"]
)

article_variety = article_variety[
    ["customer_id", "article_variety_ratio"]
]

### Preferencia por canal

In [18]:
channel_features = (
    transactions
    .assign(
        is_channel_2=(
            transactions["sales_channel_id"] == 2
        ).astype(int)
    )
    .groupby("customer_id")["is_channel_2"]
    .mean()
    .reset_index(name="channel_2_ratio")
)

### Adición de nuevas variables al dataset

In [19]:
hm_extended = (
    hm_extended
    .merge(
        basket_features,
        on="customer_id",
        how="left"
    )
    .merge(
        price_features,
        on="customer_id",
        how="left"
    )
    .merge(
        temporal_features,
        on="customer_id",
        how="left"
    )
    .merge(
        unique_articles,
        on="customer_id",
        how="left"
    )
    .merge(
        unique_product_types,
        on="customer_id",
        how="left"
    )
    .merge(
        unique_product_groups,
        on="customer_id",
        how="left"
    )
    .merge(
        unique_sections,
        on="customer_id",
        how="left"
    )
    .merge(
        unique_garment_groups,
        on="customer_id",
        how="left"
    )
    .merge(
        article_variety,
        on="customer_id",
        how="left"
    )
    .merge(
        channel_features,
        on="customer_id",
        how="left"
    )
)

In [20]:
hm_extended.head()

,customer_id,recency,frequency,monetary,avg_basket_size,std_basket_size,avg_price,std_price,avg_days_between_purchases,std_days_between_purchases,unique_articles,unique_product_types,unique_product_groups,unique_sections,unique_garment_groups,article_variety_ratio,channel_2_ratio
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,18,10,0.648983,2.100000,1.286684,0.030904,0.015718,68.666667,56.008928,19,13,4,8,9,0.904762,0.571429
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,77,23,2.601932,3.739130,3.557545,0.030255,0.016957,29.818182,32.861377,64,23,7,18,15,0.744186,0.941860
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,8,7,0.704780,2.571429,2.507133,0.039154,0.016858,121.000000,125.066382,14,10,4,9,8,0.777778,1.000000
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,472,1,0.060983,2.000000,0.000000,0.030492,0.000000,0.000000,0.000000,2,1,1,1,1,1.000000,1.000000
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,42,6,0.469695,2.166667,1.169045,0.036130,0.012638,134.000000,95.404402,12,9,5,7,6,0.923077,0.846154


In [21]:
hm_simple = pd.read_csv(
    "../../simple_datasets/hym_model_simple.csv"
)

print(
    "Clientes modelo simple:",
    hm_simple["customer_id"].nunique()
)

print(
    "Clientes modelo ampliado:",
    hm_extended["customer_id"].nunique()
)

Clientes modelo simple: 1362281
Clientes modelo ampliado: 1362281


### Guardado del dataset ampliado

In [22]:
hm_extended.to_csv(
    "../../extended_datasets/hym_model_extended.csv",
    index=False
)